# 🎼 Orchestrator-Worker with Runnable.map()

The **orchestrator-worker** workflow has an LLM **plan** the subtasks at run
time, then dispatches **one worker per subtask**. The worker count is not known
until the orchestrator has seen the input, which is exactly what separates this
pattern from parallelization.

This notebook rebuilds the report generator from
`01_orchestrator_worker_send_api.ipynb` in this module — same
planner schema and prompt, same two test topics — using `Runnable.map()` in place of
LangGraph's `Send` API.

```
topic ──▶ orchestrator ──▶ [N sections, N decided at run time]
                                │
                                ├──▶ worker ──┐
                                ├──▶ worker ──┼──▶ synthesizer ──▶ report
                                └──▶ worker ──┘
```

## Learning Objectives
In this notebook, you will learn:
1. **Runtime fan-out** - use `.map()` to run one worker per planned subtask
2. **Why not RunnableBranch** - the difference between choosing a path and spawning workers
3. **Why not RunnableParallel** - why a fixed branch set cannot express this pattern
4. **The Send comparison** - what LangGraph still does that `.map()` cannot
5. **Heterogeneous workers** - combining `RunnableBranch` inside `.map()` for mixed subtask types

## Where this fits
`01_orchestrator_worker_send_api.ipynb` built this report generator as a LangGraph graph with `Send`. This notebook shows how far plain LCEL gets with the same idea, and where it stops.

## Prerequisites
- The orchestrator-worker pattern — `01_orchestrator_worker_send_api.ipynb`
- `RunnableParallel` and `RunnableBranch` — `modules/07_parallelization/notebooks/03_parallelization_lcel.ipynb`, `modules/06_routing/notebooks/03_routing_lcel_runnable_branch.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📦 Step 1: Environment Setup

Load the API key and create the chat model through the course factory. `temperature=0.7` because the workers write prose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API Keys & Import Dependencies
# ============================================================================
# We use python-dotenv to securely load API keys from a .env file
# This is a best practice - never hardcode API keys in your notebooks!
# ============================================================================

import time
import warnings

from aied_helpers import get_openai_llm
from dotenv import find_dotenv, load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# The same model plans the report and writes each section.
llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print("✅ Environment variables loaded successfully!")
print(f"🤖 LLM initialized: {llm.model_name}")

---

## 🧭 Part 1: Why the Two Primitives You Know Do Not Work

Before reaching for the right tool, it is worth seeing why the familiar two fail.
This cell uses no LLM — it feeds a five-item plan to each primitive and counts
what ran.

- **`RunnableBranch`** inspects the plan and runs **one** thing, once. It is a
  router. It has no notion of "per item".
- **`RunnableParallel`** runs a branch set you named in the source code. You
  cannot name branches for sections that do not exist yet.

In [ ]:
# ============================================================================
# NEGATIVE EXAMPLE: What RunnableBranch does with a variable-length plan
# ============================================================================
fake_plan = [f"section {i}" for i in range(5)]

branch_on_plan = RunnableBranch(
    (lambda plan: len(plan) > 3, RunnableLambda(lambda plan: "big-report branch ran ONCE")),
    RunnableLambda(lambda plan: "small-report branch ran ONCE"),
)

print(f"Plan has {len(fake_plan)} sections")
print(f"RunnableBranch result: {branch_on_plan.invoke(fake_plan)}")
print("❌ One branch ran once. We needed 5 workers.")

---

## 🗺️ Part 2: The Orchestrator

Identical to the LangGraph version: a Pydantic schema plus
`with_structured_output` forces the planner to return a list of sections, and
the system prompt tells it to size the plan to the topic rather than pad it.

The length of `plan.sections` is the number of workers we will need. Nothing in
the code decides it.

In [ ]:
# ============================================================================
# PLAN SCHEMA: What the orchestrator must produce
# ============================================================================


class Section(BaseModel):
    """A single report section identified by the orchestrator."""

    name: str = Field(description="Short title of this report section")
    description: str = Field(
        description="What this section should cover - scope, angle, key points to hit"
    )


class ReportPlan(BaseModel):
    """The orchestrator's full breakdown of a topic into sections."""

    sections: list[Section] = Field(
        description=(
            "The sections needed to cover this topic well - can be as few or as "
            "many as the topic warrants"
        )
    )


orchestrator = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a report planning orchestrator. Given a topic, plan the sections "
            "of a short report. Create one section for each distinct question the topic "
            "actually asks. A narrow topic with a single factual answer needs very few "
            "sections. A broad topic that spans several fields gets a separate section "
            "for each field and each major sub-question. Never add Introduction, "
            "Conclusion, Summary or other padding sections.",
        ),
        ("human", "Plan the sections for a report on: {topic}"),
    ]
) | llm.with_structured_output(ReportPlan)

print("✅ Orchestrator ready")

---

## 👷 Part 3: The Worker

One worker writes one section. It is an ordinary chain — the important part is
that it takes a **single** section, not the list. `.map()` will handle the
repetition.

In [ ]:
# ============================================================================
# WORKER CHAIN: Writes exactly ONE section
# ============================================================================
worker_chain = (
    ChatPromptTemplate.from_template(
        "Write the '{section_name}' section of a report on: {topic}\n\n"
        "This section should specifically cover: {section_description}\n\n"
        "Write 2-4 focused paragraphs of body text only. Do not include a title "
        "or heading line (one will be added separately) and do not repeat content "
        "that belongs in other sections."
    )
    | llm
    | StrOutputParser()
)


def write_section(task: dict) -> str:
    """Run one worker and prefix the section heading."""
    body = worker_chain.invoke(task)
    return f"## {task['section_name']}\n\n{body}"


worker = RunnableLambda(write_section)

print("✅ Worker defined — it handles ONE section per call")

---

## 🚀 Part 4: The Fan-Out: Runnable.map()

`.map()` converts a runnable that takes **one item** into a runnable that takes
a **list** and returns a **list**, running the items concurrently through the
same thread pool `RunnableParallel` uses.

This is the closest thing LangChain has to LangGraph's `Send`. The list length is
whatever the orchestrator produced, so the worker count is decided at run time.

> **Key insight**: `RunnableParallel` fans out over **branch names you wrote**.
> `.map()` fans out over **list items the input produced**. That single
> difference is the whole pattern.

In [ ]:
# ============================================================================
# PIPELINE: Orchestrate, fan out with .map(), then synthesize
# ============================================================================


def plan_to_tasks(payload: dict) -> list[dict]:
    """Turn the orchestrator's plan into one task dict per worker."""
    plan = payload["plan"]
    print(
        f"🗺️  Orchestrator planned {len(plan.sections)} section(s): "
        f"{[s.name for s in plan.sections]}"
    )
    return [
        {
            "topic": payload["topic"],
            "section_name": section.name,
            "section_description": section.description,
        }
        for section in plan.sections
    ]


def synthesize(sections: list[str]) -> str:
    """Fan-in: merge every worker's output into one report."""
    return "\n\n".join(sections)


report_chain = (
    RunnableLambda(lambda payload: {"topic": payload["topic"], "plan": orchestrator.invoke(payload)})
    | RunnableLambda(plan_to_tasks)
    | worker.map()                      # ← the dynamic fan-out
    | RunnableLambda(synthesize)
)

print("✅ Report chain assembled")

---

## 🧪 Part 5: Proving the Dynamism

The same two test topics as the LangGraph notebook: one trivial, one broad. If
the pattern works, the worker count differs between them without any code
change.

In [ ]:
# ============================================================================
# TEST: Worker count should vary with topic complexity
# ============================================================================
test_topics = [
    "What is the boiling point of water at sea level?",                  # narrow
    "A buyer's guide to electric cars: purchase cost, range, charging, "
    "maintenance, resale value and government incentives",               # broad
]

for topic in test_topics:
    print(f"\n{'=' * 70}")
    print(f"TOPIC: {topic}")
    print("=" * 70)

    start = time.perf_counter()
    report = report_chain.invoke({"topic": topic})
    elapsed = time.perf_counter() - start

    print(f"\n⏱️  Wall time: {elapsed:.2f}s")
    print(f"📄 Sections written: {sum(line.startswith('## ') for line in report.splitlines())}")
    print(f"\nReport preview:\n{report[:400]}...")

---

## ⚖️ Part 6: What .map() Gives Up Versus Send

`.map()` covers the dynamic fan-out and fan-in faithfully, but it is not a full
replacement for `Send`. Know the four gaps before choosing.

| | LangGraph `Send` | LCEL `.map()` |
|---|---|---|
| Worker identity | each `Send` names its own target node | every item goes to the **same** runnable |
| Results land in | shared graph state, merged by a reducer (in `Send` order) | an **ordered list**, positionally matched to the input |
| One worker fails | the graph can route around it | the whole batch raises, unless you use `batch(return_exceptions=True)` |
| Durability | per-worker checkpoints, interrupts, resume | none — the chain is stateless |

Both keep the input order in the final result: `.map()` returns a list in input
order, and LangGraph's reducer merges parallel writes in `Send` order (01 checked
this). What only LangGraph shows you is the **completion** order, through
`stream_mode="updates"`.

In [ ]:
# ============================================================================
# ERROR SEMANTICS: One failing worker aborts the whole .map() batch
# ============================================================================
flaky = RunnableLambda(lambda i: 1 / 0 if i == 1 else f"ok-{i}")

try:
    flaky.map().invoke([0, 1, 2])
except ZeroDivisionError:
    print("❌ .map() propagated the worker failure and lost the good results")

# batch() with return_exceptions keeps the successes and reports the failure.
print(f"✅ batch(return_exceptions=True): {flaky.batch([0, 1, 2], return_exceptions=True)}")

---

## 🎭 Part 7: Heterogeneous Workers

The "same runnable for every item" limit is softer than it looks. Put a
`RunnableBranch` **inside** the worker and each item picks its own specialist,
while `.map()` still does the fan-out.

This composition is the closest LCEL gets to `Send` targeting different nodes.

In [ ]:
# ============================================================================
# MIXED WORKERS: RunnableBranch inside .map() routes each subtask
# ============================================================================
mixed_worker = RunnableBranch(
    (lambda t: t["kind"] == "code", RunnableLambda(lambda t: f"[coder] {t['task']}")),
    (lambda t: t["kind"] == "research", RunnableLambda(lambda t: f"[researcher] {t['task']}")),
    RunnableLambda(lambda t: f"[writer] {t['task']}"),
)

mixed_plan = [
    {"kind": "code", "task": "implement the parser"},
    {"kind": "research", "task": "survey prior art"},
    {"kind": "prose", "task": "write the intro"},
]

for line in mixed_worker.map().invoke(mixed_plan):
    print(f"  {line}")

---

## 📝 Summary

We rebuilt the LangGraph orchestrator-worker report generator in LCEL.

### 1. The translation table

| LangGraph | LCEL |
|---|---|
| `Send("llm_call", {...})` per section | `worker.map()` over a list of task dicts |
| `add_conditional_edges(node, assign_workers, [...])` | piping the plan straight into `.map()` |
| `Annotated[list, operator.add]` collecting worker output | the ordered list `.map()` returns |
| a `synthesizer` node | a `RunnableLambda` after the map |
| `WorkerState` TypedDict | the plain task dict each worker receives |

### 2. The three fan-out primitives, side by side

| Primitive | Workers that run | Who decides the count |
|---|---|---|
| `RunnableBranch` | 1 of a fixed set | nobody — always one |
| `RunnableParallel` | all of a fixed set | you, when writing the code |
| `.map()` | the same worker, N times | the input, at run time |

### 3. When to stay on LangGraph
Choose `Send` over `.map()` when workers must write into shared state, when
different subtasks need genuinely different **nodes** rather than a branch inside
one worker, or when you need per-worker checkpointing, interrupts and resume.

### Next Steps
- **`modules/09_evaluator_optimizer/notebooks/01_evaluator_optimizer_langgraph.ipynb`**: the last workflow pattern — generate, critique, and loop until the output passes